# Preprocessing

## Data Loading & Train/Validation/Test Split

In [126]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os

In [127]:
data_path = '../../data/raw/AmesHousing.csv'
df = pd.read_csv(data_path)

In [128]:
X = df.drop(columns=["SalePrice"])
y = df["SalePrice"]

# 1) separar treino e temp (validação + teste) com 70% dos dados para treino e 30% para temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

# 2) separar validação e teste com 50% dos dados para cada
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)


print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (2051, 81) (2051,)
Validation: (439, 81) (439,)
Test: (440, 81) (440,)


In [129]:
train, validation, test =  df.loc[X_train.index].copy(), df.loc[X_val.index].copy(), df.loc[X_test.index].copy()

## Missing Value Handling

### Structural

Variables where `NaN` means absence of the feature (e.g no Pool, no Fence, no Fireplace).

- NaN values in Categorical features -> 'None'
- NaN values in Numerical features -> 0

---

I can't confirm if `Mas Vnr Area` is missing because the house doesn't have a Masonry Veneer, or if it's because the data is missing because of an input error. I am choosing to treat it as Structural for simplification reasons.

Since this is the approach I'm taking for the `Mas Vnr Area`, I'm going to assume the same Structural handling for `Mas Vnr Type`, since they're highly related in their missing values.

---

For the following Basement grouping:
- BsmtFin SF 1 , BsmtFin SF 2
- Bsmt Unf SF
- Total Bsmt SF
- Bsmt Full Bath
- Bsmt Half Bath

They have few missing values and it always happens at the same time within this group, altough I can't be sure if it means there is no Basement or if it is a mistake in the input of the rows, I will treat it as a Structural missingness because it is not significant on the dataset and for simplification reasons.

In [130]:
structural_categorical_features = [
    "Alley",
    'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
    'BsmtFin Type 1', 'BsmtFin Type 2',
    'Fireplace Qu',
    'Garage Type', 'Garage Finish',
    'Garage Qual','Garage Cond',
    'Pool QC', 'Fence', 'Misc Feature',
    'Mas Vnr Type'
]

structural_numerical_features = [
    'Garage Yr Blt', 'Garage Cars', 'Garage Area',
    'BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF',
    'Total Bsmt SF', 'Bsmt Full Bath', 'Bsmt Half Bath',
    'Mas Vnr Area' 
]

In [131]:
def fill_structural_features(train, validation, test, structural_categorical_features, structural_numerical_features):

    """This function fills missing values in structural features with 
        - 'None' for categorical features
        - 0 for numerical features
    """

    for col in structural_categorical_features:
        train[col] = train[col].fillna("None")
        validation[col] = validation[col].fillna("None")
        test[col] = test[col].fillna("None")

    for col in structural_numerical_features:
        train[col] = train[col].fillna(0)
        validation[col] = validation[col].fillna(0)
        test[col] = test[col].fillna(0)

    print("Structural features filled with 'None' for categorical and 0 for numerical.")

In [132]:
fill_structural_features(train, validation, test, structural_categorical_features, structural_numerical_features)

Structural features filled with 'None' for categorical and 0 for numerical.


### Non-Structural

In this dataset there are two variables we have considered as Non-Structural.

- **Lot Frontage** - with missing values highly related to other variables like Neighbourhood, Lot Area, Lot Shape, Lot Config and MS SubClass
- **Electrical** - with a single missing value that appears to be completely random

For these missing values I am:
- Going to input **Lot Frontage** witht the median of the Neighbourhood as it makes the most sense as to why it's related.
- Going to imput **Electrical** with the mode.

In [133]:
# Calculate median Lot Frontage for each neighborhood in the training set
neighborhood_medians = train.groupby('Neighborhood')['Lot Frontage'].median()

global_median = train['Lot Frontage'].median() # Fallback in case a neighborhood in val/test is not in train

def impute_lot_frontage(df, neighborhood_medians, global_median):
    """
    Imputes missing values in target_col using the median value 
    of the neighborhood_col from the training set.
    """
    df = df.copy()
    
    impute_map = df['Neighborhood'].map(neighborhood_medians) # Map neighborhood to its median Lot Frontage
    
    # Fill missing Lot Frontage with neighborhood median, then global median if still missing
    df['Lot Frontage'] = df['Lot Frontage'].fillna(impute_map).fillna(global_median) 

    return df

In [134]:
test = impute_lot_frontage(test, neighborhood_medians, global_median)
validation = impute_lot_frontage(validation, neighborhood_medians, global_median)
train = impute_lot_frontage(train, neighborhood_medians, global_median)

In [135]:
eletrical_mode = train['Electrical'].mode()[0]

def impute_electrical(df, mode_value):
    df = df.copy()
    df['Electrical'] = df['Electrical'].fillna(mode_value)
    return df

train = impute_electrical(train, eletrical_mode)
validation = impute_electrical(validation, eletrical_mode)
test = impute_electrical(test, eletrical_mode)

## Feature Engineering

In this section I will create a few new features that might be useful for our models.

- **Binary Has Flags** - In our correlation analysis we can see that SalePrice is highly affected wether a house has a feature or not.
    - Has Fireplace: 1 if Fireplaces > 0, else 0
    - Has Garage: 1 if Garage Area > 0, else 0

- **Bathroom Counts** - Real estate listings usually summarize bathrooms into a single number.
    - Total Bath: Full Bath + (0.5 * Half Bath) + Bsmt Full Bath + (0.5 * Bsmt Half Bath).

- **Simplified Quality & Condition** - Many features use the same scale.
    - Overall Score: Overall Qual * Overall Cond
    - Garage Score: Garage Qual * Garage Cond
    - Kitchen Score : KitchenQual * Kitchen

- **Modernity and Age Features**
    - House Age: Yr Sold - Year Built
    - Years Since Remodel: YrSold - YearRemodAdd

- **Total Square Feet** - Real estate listings usually show the total Square feet of the property
    - TotalSF : Gr Liv Area + Total Bsmt SF

- **Neighbourhood Rank** - Different neighbourhoods can be more expensive depending on the location

Since SalePrice is extremely skewed to the right (as most house prices are) I'm going to apply a log on it.

In [136]:
garage_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'NA': 0, 'None': 0}

neighbor_rank = train.groupby('Neighborhood')['SalePrice'].median().sort_values().index
neighbor_map = {k: i for i, k in enumerate(neighbor_rank)}

for df_ in [train, validation, test]:
    
    df_['Has Garage'] = (df_['Garage Area'] > 0).astype(int)

    df_['Has Fireplace'] = (df_['Fireplaces'] > 0).astype(int)

    df_['Total Bath'] = df_['Full Bath'] + (0.5 * df_['Half Bath']) + df_['Bsmt Full Bath'] + (0.5 * df_['Bsmt Half Bath'])

    df_['Overall Score'] = (df_['Overall Qual'] + df_['Overall Cond']).astype(int)

    df_['Garage Score'] = df_['Garage Qual'].map(garage_map) + df_['Garage Cond'].map(garage_map)

    df_['Kitchen Score'] = (df_['Kitchen Qual'].map(garage_map) * df_['Kitchen AbvGr']).astype(int)

    df_['House Age'] = (df_['Yr Sold'] - df_['Year Built']).clip(lower=0)
    
    df_['Years Since Remodel'] = (df_['Yr Sold'] - df_['Year Remod/Add']).clip(lower=0)

    df_['TotalSF'] = df_['Gr Liv Area'] + df_['Total Bsmt SF']

    df_['Neighborhood Rank'] = df_['Neighborhood'].map(neighbor_map)

    df_['Log_SalePrice'] = np.log(df_['SalePrice'])

In [137]:
train[['Has Garage', 'Has Fireplace', 'Total Bath', 'Overall Score', 'Garage Score', 'Kitchen Score',
       'House Age', 'Years Since Remodel', 'TotalSF', 'Neighborhood Rank', 'Log_SalePrice']].describe()

,Has Garage,Has Fireplace,Total Bath,Overall Score,Garage Score,Kitchen Score,House Age,Years Since Remodel,TotalSF,Neighborhood Rank,Log_SalePrice
count,2051.000000,2051.000000,2051.000000,2051.000000,2051.000000,2051.000000,2051.000000,2051.000000,2051.000000,2051.000000,2051.000000
mean,0.946368,0.504632,2.203559,11.648952,5.610922,3.628474,37.803998,23.980985,2546.074598,12.657728,12.011146
std,0.225346,0.500100,0.817570,1.707836,1.409292,0.847060,30.446813,20.755044,816.889356,7.361725,0.403107
min,0.000000,0.000000,1.000000,3.000000,0.000000,0.000000,0.000000,0.000000,334.000000,0.000000,9.456341
25%,1.000000,0.000000,1.500000,11.000000,6.000000,3.000000,9.000000,5.000000,1997.000000,7.000000,11.767568
50%,1.000000,1.000000,2.000000,12.000000,6.000000,3.000000,36.000000,16.000000,2446.000000,12.000000,11.982929
75%,1.000000,1.000000,2.500000,13.000000,6.000000,4.000000,55.000000,43.000000,2980.000000,19.000000,12.254863
max,1.000000,1.000000,7.000000,19.000000,10.000000,9.000000,136.000000,60.000000,11752.000000,27.000000,13.534473


## Simplifying Categorical Columns

In [138]:
def neighborhood_simplify(train, val, test):
   
    train_temp = train.copy()
    train_temp['PricePerSF'] = train_temp['SalePrice'] / train_temp['Gr Liv Area']
    medians = train_temp.groupby('Neighborhood')['PricePerSF'].median()
    
    low_cutoff = medians.quantile(0.33)
    high_cutoff = medians.quantile(0.66)
    
    def map_to_tier(name):
        
        val = medians.get(name, medians.median()) 
        
        if val <= low_cutoff:
            return 'Tier_1_Budget'
        elif val <= high_cutoff:
            return 'Tier_2_Mid'
        else:
            return 'Tier_3_Luxury'

    for df in [train, val, test]:
        df['Neighborhood_Simplified'] = df['Neighborhood'].apply(map_to_tier)
    
    return train, val, test

train, validation, test = neighborhood_simplify(train, validation, test)

In [139]:
def simplify_conditions(df):
    df = df.copy()
    
    positives = ['PosN', 'PosA']
    negatives = ['Artery', 'Feedr', 'RRNn', 'RRAn', 'RRNe', 'RRAe']
    
    def classify(row):
        # Check both columns for a positive condition first
        if row['Condition 1'] in positives or row['Condition 2'] in positives:
            return 'Positive'
        # Then check if either is a negative condition
        elif row['Condition 1'] in negatives or row['Condition 2'] in negatives:
            return 'Negative'
        # Otherwise, it's normal
        else:
            return 'Normal'

    df['Condition_Group'] = df.apply(classify, axis=1)
    df = df.drop(['Condition 1', 'Condition 2'], axis=1)
    
    return df

train = simplify_conditions(train)
validation = simplify_conditions(validation)
test = simplify_conditions(test)

## Price Tier Target for Classification Tasks

In [140]:
_, bins = pd.qcut(train['SalePrice'], q=3, retbins=True, labels=False)
bins[0] = -np.inf
bins[-1] = np.inf

for df_ in [train, validation, test]:
    df_['Price Tier'] = pd.cut(df_['SalePrice'], bins=bins, labels=['Low', 'Medium', 'High'])
    df_['Is High Price'] = (df_['Price Tier'] == 'High').astype(int)

In [141]:
train['Price Tier'].value_counts()

Price Tier
Low       695
High      682
Medium    674
Name: count, dtype: int64

In [142]:
train['Is High Price'].value_counts(normalize=True)

Is High Price
0    0.667479
1    0.332521
Name: proportion, dtype: float64

## Final Verifications

In [143]:
def check_missing_values(df):
    missing = df.isnull().sum()
    return missing[missing > 0]

In [144]:
print("------ Checking for missing values in each set -----")

for name, df_ in [("Train", train), ("Validation", validation), ("Test", test)]:
    
    missing = check_missing_values(df_)
    
    if missing.empty:
        print(f"{name} - No Missing values.")
    else:        
        print(f"{name} - Missing values:\n{missing}")

------ Checking for missing values in each set -----
Train - No Missing values.
Validation - No Missing values.
Test - No Missing values.


## Exporting Datasets

In [145]:
os.makedirs('../../data/processed', exist_ok=True)

train.to_csv('../../data/processed/train.csv', index=False)
validation.to_csv('../../data/processed/val.csv', index=False)
test.to_csv('../../data/processed/test.csv', index=False)